In [7]:
import pandas as pd
import numpy as np
import os

print("="*50)
print("📥 DOWNLOADING ESSENTIAL MIMIC-IV TABLES")
print("="*50)

# Create data folder
os.makedirs("../data/mimic-iv-core", exist_ok=True)

# We'll use a publicly available mirror with sample data
# These are the exact tables we need for the hackathon

print("\n📊 Creating sample MIMIC-IV tables...")

# 1. PATIENTS table
patients_data = {
    'subject_id': list(range(10000001, 10000101)),
    'gender': ['M', 'F', 'F', 'M', 'F'] * 20,
    'anchor_age': [65, 72, 58, 81, 45, 67, 53, 79, 62, 71] * 10,
    'anchor_year': [2015, 2014, 2016, 2013, 2017] * 20,
    'dod': [None] * 80 + ['2015-03-15', '2014-11-22'] * 10
}
patients_df = pd.DataFrame(patients_data)
patients_df.to_csv("../data/mimic-iv-core/patients.csv", index=False)
print("✅ Created patients.csv (100 patients)")

# 2. ADMISSIONS table
admissions_data = {
    'subject_id': patients_df['subject_id'].values,
    'hadm_id': list(range(20000001, 20000101)),
    'admittime': pd.date_range('2014-01-01', periods=100, freq='3D'),
    'dischtime': pd.date_range('2014-01-05', periods=100, freq='3D'),
    'admission_type': ['EMERGENCY'] * 70 + ['ELECTIVE'] * 30,
    'insurance': ['Medicare'] * 50 + ['Private'] * 30 + ['Medicaid'] * 20,
    'marital_status': ['MARRIED'] * 40 + ['SINGLE'] * 30 + ['WIDOWED'] * 30,
    'ethnicity': ['WHITE'] * 60 + ['BLACK/AFRICAN AMERICAN'] * 20 + ['HISPANIC/LATINO'] * 20
}
admissions_df = pd.DataFrame(admissions_data)
admissions_df.to_csv("../data/mimic-iv-core/admissions.csv", index=False)
print("✅ Created admissions.csv (100 admissions)")

# 3. ICU STAYS table
icustays_data = {
    'subject_id': admissions_df['subject_id'].values,
    'hadm_id': admissions_df['hadm_id'].values,
    'stay_id': list(range(30000001, 30000101)),
    'first_careunit': ['Medical ICU'] * 60 + ['Surgical ICU'] * 20 + ['Cardiac ICU'] * 20,
    'last_careunit': ['Medical ICU'] * 60 + ['Surgical ICU'] * 20 + ['Cardiac ICU'] * 20,
    'intime': admissions_df['admittime'].values,
    'outtime': admissions_df['dischtime'].values,
    'los': [4.5] * 100
}
icustays_df = pd.DataFrame(icustays_data)
icustays_df.to_csv("../data/mimic-iv-core/icustays.csv", index=False)
print("✅ Created icustays.csv (100 ICU stays)")

# 4. D_ITEMS table (vital sign mappings)
d_items_data = {
    'itemid': [220045, 220179, 220180, 220277, 220210, 223762, 223761, 220050],
    'label': ['Heart Rate', 'Non Invasive Blood Pressure systolic', 
              'Non Invasive Blood Pressure diastolic', 'O2 saturation pulseoxymetry',
              'Respiratory Rate', 'Temperature Fahrenheit', 'Temperature Celsius',
              'Arterial Blood Pressure systolic'],
    'abbreviation': ['HR', 'NBP_S', 'NBP_D', 'SpO2', 'RR', 'Temp_F', 'Temp_C', 'ABP_S'],
    'category': ['Routine Vital Signs'] * 8,
    'unitname': ['bpm', 'mmHg', 'mmHg', '%', 'insp/min', '°F', '°C', 'mmHg']
}
d_items_df = pd.DataFrame(d_items_data)
d_items_df.to_csv("../data/mimic-iv-core/d_items.csv", index=False)
print("✅ Created d_items.csv (vital sign mappings)")

# 5. CHARTEVENTS table (vital signs - this will be big!)
print("📊 Generating chartevents.csv...")

np.random.seed(42)
n_measurements = 5000  # 5000 vital sign measurements

# Create base dataframe
chartevents_list = []
base_time = pd.Timestamp('2014-01-01 00:00:00')

for i in range(n_measurements):
    chartevents_list.append({
        'subject_id': np.random.choice(icustays_df['subject_id']),
        'hadm_id': np.random.choice(icustays_df['hadm_id']),
        'stay_id': np.random.choice(icustays_df['stay_id']),
        'charttime': base_time + pd.Timedelta(minutes=15 * i),
        'itemid': np.random.choice(d_items_df['itemid']),
    })

chartevents_df = pd.DataFrame(chartevents_list)

# Add realistic values based on itemid
conditions = [
    (chartevents_df['itemid'] == 220045),  # Heart Rate
    (chartevents_df['itemid'] == 220179),  # Systolic BP
    (chartevents_df['itemid'] == 220180),  # Diastolic BP
    (chartevents_df['itemid'] == 220277),  # SpO2
    (chartevents_df['itemid'] == 220210),  # Respiratory Rate
]
choices = [
    np.random.normal(80, 15, n_measurements),   # HR ~80
    np.random.normal(120, 20, n_measurements),  # SBP ~120
    np.random.normal(70, 10, n_measurements),   # DBP ~70
    np.random.normal(97, 3, n_measurements),    # SpO2 ~97
    np.random.normal(16, 4, n_measurements),    # RR ~16
]
chartevents_df['valuenum'] = np.select(conditions, choices, default=np.nan)

# Clip to realistic ranges
chartevents_df.loc[chartevents_df['itemid'] == 220045, 'valuenum'] = chartevents_df.loc[chartevents_df['itemid'] == 220045, 'valuenum'].clip(40, 150)
chartevents_df.loc[chartevents_df['itemid'] == 220179, 'valuenum'] = chartevents_df.loc[chartevents_df['itemid'] == 220179, 'valuenum'].clip(60, 200)
chartevents_df.loc[chartevents_df['itemid'] == 220180, 'valuenum'] = chartevents_df.loc[chartevents_df['itemid'] == 220180, 'valuenum'].clip(30, 120)
chartevents_df.loc[chartevents_df['itemid'] == 220277, 'valuenum'] = chartevents_df.loc[chartevents_df['itemid'] == 220277, 'valuenum'].clip(80, 100)
chartevents_df.loc[chartevents_df['itemid'] == 220210, 'valuenum'] = chartevents_df.loc[chartevents_df['itemid'] == 220210, 'valuenum'].clip(8, 30)

chartevents_df['valueuom'] = 'bpm'
chartevents_df.to_csv("../data/mimic-iv-core/chartevents.csv", index=False)
print(f"✅ Created chartevents.csv ({len(chartevents_df):,} vital sign measurements)")

# 6. LABEVENTS table
print("📊 Generating labevents.csv...")
lab_items = {
    50813: ('Lactate', 1.5, 0.8, 0.5, 5.0),
    50912: ('Creatinine', 1.0, 0.3, 0.5, 2.0),
    51301: ('WBC', 10.0, 4.0, 4.0, 20.0),
    50811: ('Hemoglobin', 12.0, 2.0, 8.0, 16.0),
    50820: ('pH', 7.4, 0.1, 7.2, 7.6)
}

n_labs = 2000
labevents_list = []

for i in range(n_labs):
    itemid = np.random.choice(list(lab_items.keys()))
    mean, std, min_val, max_val = lab_items[itemid][1:]
    value = np.random.normal(mean, std)
    value = np.clip(value, min_val, max_val)
    
    labevents_list.append({
        'subject_id': np.random.choice(icustays_df['subject_id']),
        'hadm_id': np.random.choice(icustays_df['hadm_id']),
        'charttime': base_time + pd.Timedelta(hours=2 * i),
        'itemid': itemid,
        'valuenum': value
    })

labevents_df = pd.DataFrame(labevents_list)
labevents_df.to_csv("../data/mimic-iv-core/labevents.csv", index=False)
print(f"✅ Created labevents.csv ({len(labevents_df):,} lab measurements)")

# 7. DIAGNOSES_ICD table (for labels)
print("📊 Creating diagnoses_icd.csv...")
n_diagnoses = 300
deterioration_codes = ['A41', 'R65.2', 'J96', 'I46']  # Sepsis, Respiratory failure, Cardiac arrest
other_codes = ['I10', 'E11', 'J44', 'I25', 'N18']  # Hypertension, Diabetes, COPD, CAD, CKD

diagnoses_list = []
for i in range(n_diagnoses):
    # 30% chance of deterioration code
    if np.random.random() < 0.3:
        code = np.random.choice(deterioration_codes)
    else:
        code = np.random.choice(other_codes)
    
    diagnoses_list.append({
        'subject_id': np.random.choice(icustays_df['subject_id']),
        'hadm_id': np.random.choice(icustays_df['hadm_id']),
        'icd_code': code,
        'icd_version': 10
    })

diagnoses_df = pd.DataFrame(diagnoses_list)
diagnoses_df.to_csv("../data/mimic-iv-core/diagnoses_icd.csv", index=False)
print(f"✅ Created diagnoses_icd.csv ({len(diagnoses_df)} diagnoses)")

print("\n" + "="*50)
print("🎉 ALL ESSENTIAL TABLES CREATED!")
print("="*50)
print(f"\n📁 Data saved to: data/mimic-iv-core/")
print("\n📊 Summary:")
print(f"   - 100 patients")
print(f"   - 100 ICU stays")
print(f"   - {len(chartevents_df):,} vital sign measurements")
print(f"   - {len(labevents_df):,} lab results")
print(f"   - {len(diagnoses_df)} diagnoses")
print(f"\n💡 This is a SYNTHETIC dataset based on MIMIC-IV patterns")
print(f"   Perfect for hackathon development and demo!")
print("\n🚀 Ready to run the pipeline!")

📥 DOWNLOADING ESSENTIAL MIMIC-IV TABLES

📊 Creating sample MIMIC-IV tables...
✅ Created patients.csv (100 patients)
✅ Created admissions.csv (100 admissions)
✅ Created icustays.csv (100 ICU stays)
✅ Created d_items.csv (vital sign mappings)
📊 Generating chartevents.csv...
✅ Created chartevents.csv (5,000 vital sign measurements)
📊 Generating labevents.csv...
✅ Created labevents.csv (2,000 lab measurements)
📊 Creating diagnoses_icd.csv...
✅ Created diagnoses_icd.csv (300 diagnoses)

🎉 ALL ESSENTIAL TABLES CREATED!

📁 Data saved to: data/mimic-iv-core/

📊 Summary:
   - 100 patients
   - 100 ICU stays
   - 5,000 vital sign measurements
   - 2,000 lab results
   - 300 diagnoses

💡 This is a SYNTHETIC dataset based on MIMIC-IV patterns
   Perfect for hackathon development and demo!

🚀 Ready to run the pipeline!
